In [ ]:
import ROOT
import math
import sys
from array import array

def BinomErr(n, k):
    """Calculates the binomial error."""
    if n <= 0.0:
        return 0.0
    p = float(k) / n
    return math.sqrt(max(0.0, p * (1.0 - p) / n))

def PrintEff(label, den, num):
    """Prints the calculated efficiency and its binomial error."""
    eff = float(num) / den if den > 0.0 else 0.0
    print(f"{label}: {num} / {den} = {eff} +/- {BinomErr(den, num)}")

def BindDouble(tree, name, var_array, required=True):
    """Helper to bind a double branch to an array."""
    if not tree.GetBranch(name):
        if required:
            print(f"ERROR: missing required branch: {name}", file=sys.stderr)
        return False
    tree.SetBranchStatus(name, 1)
    tree.SetBranchAddress(name, var_array)
    return True

def BindInt(tree, name, var_array, required=True):
    """Helper to bind an integer branch to an array."""
    if not tree.GetBranch(name):
        if required:
            print(f"ERROR: missing required branch: {name}", file=sys.stderr)
        return False
    tree.SetBranchStatus(name, 1)
    tree.SetBranchAddress(name, var_array)
    return True

def LowQ2ScatteredElectronReach_FromTuple(
    tupleFile,
    outFile="LowQ2ScatteredElectronReach.root",
    etaCut=-4.0,
    requireDecayLeptonsCentral=False
):
    """
    Computes the simple truth-level geometric reach of the scattered electron to
    the low-Q2 tagger from the flattened UpsilonTree.
    """
    fin = ROOT.TFile.Open(tupleFile, "READ")
    if not fin or fin.IsZombie():
        print(f"ERROR: could not open input file: {tupleFile}", file=sys.stderr)
        return

    tr = fin.Get("UpsilonTree")
    if not tr:
        print("ERROR: input file does not contain TTree 'UpsilonTree'.", file=sys.stderr)
        if fin: fin.Close()
        return

    # Disable all branches, then enable only what we need for performance
    tr.SetBranchStatus("*", 0)

    # Use arrays to link to the tree branches
    true_scat_eta = array('d', [-999.0])
    true_scat_E   = array('d', [-999.0])
    true_scat_pT  = array('d', [-999.0])
    true_eplus_eta = array('d', [-999.0])
    true_decay_eminus_eta = array('d', [-999.0])
    true_Q2 = array('d', [-999.0])
    true_W  = array('d', [-999.0])
    true_yUpsilon = array('d', [-999.0])
    reco_topology = array('i', [0])

    ok = True
    ok &= BindDouble(tr, "true_scat_eMinus_eta", true_scat_eta, True)
    ok &= BindDouble(tr, "true_scat_eMinus_E", true_scat_E, True)
    ok &= BindDouble(tr, "true_scat_eMinus_pT", true_scat_pT, True)
    ok &= BindDouble(tr, "true_ePlus_eta", true_eplus_eta, True)
    ok &= BindDouble(tr, "true_decay_eMinus_eta", true_decay_eminus_eta, True)
    ok &= BindDouble(tr, "true_Q2", true_Q2, True)
    ok &= BindDouble(tr, "true_W", true_W, True)
    ok &= BindDouble(tr, "true_y", true_yUpsilon, True)
    hasRecoTopology = BindInt(tr, "reco_topology", reco_topology, False)

    if not ok:
        print("ERROR: required branches are missing. Run this on the flattened UpsilonTree output.", file=sys.stderr)
        fin.Close()
        return

    fout = ROOT.TFile.Open(outFile, "RECREATE")
    if not fout or fout.IsZombie():
        print(f"ERROR: could not create output file: {outFile}", file=sys.stderr)
        fin.Close()
        return

    # Variable Q2 bins emphasize the low-Q2 region without losing high-Q2 context.
    q2Edges_list = [0.0, 0.01, 0.02, 0.05, 0.10, 0.20, 0.50, 1.0, 2.0, 5.0, 10.0, 20.0, 50.0]
    q2Edges = array('d', q2Edges_list)
    nQ2 = len(q2Edges_list) - 1

    eff_vs_Q2  = ROOT.TEfficiency("eff_lowQ2Reach_vs_trueQ2", "Truth scattered e^{-} reaches low-Q^{2} tagger;true Q^{2} [GeV^{2}];efficiency", nQ2, q2Edges)
    eff_vs_W   = ROOT.TEfficiency("eff_lowQ2Reach_vs_trueW", "Truth scattered e^{-} reaches low-Q^{2} tagger;true W [GeV];efficiency", 75, 0.0, 150.0)
    eff_vs_yVM = ROOT.TEfficiency("eff_lowQ2Reach_vs_trueUpsilonY", "Truth scattered e^{-} reaches low-Q^{2} tagger;true #Upsilon rapidity;efficiency", 50, -5.0, 5.0)
    eff_vs_E   = ROOT.TEfficiency("eff_lowQ2Reach_vs_trueScatE", "Truth scattered e^{-} reaches low-Q^{2} tagger;true scattered-e energy [GeV];efficiency", 75, 0.0, 25.0)
    eff_vs_pT  = ROOT.TEfficiency("eff_lowQ2Reach_vs_trueScatPT", "Truth scattered e^{-} reaches low-Q^{2} tagger;true scattered-e p_{T} [GeV];efficiency", 75, 0.0, 10.0)

    h_eta_den = ROOT.TH1D("h_trueScatEta_den", "Truth scattered e^{-} #eta, denominator;#eta;events", 120, -12.0, 3.0)
    h_eta_num = ROOT.TH1D("h_trueScatEta_num_lowQ2Reach", "Truth scattered e^{-} #eta, reaches low-Q^{2} tagger;#eta;events", 120, -12.0, 3.0)

    h_summary = ROOT.TH1D("h_lowQ2Reach_summary", "Low-Q^{2} scattered-electron reach summary", 8, 0.5, 8.5)
    h_summary.GetXaxis().SetBinLabel(1, "truth denom")
    h_summary.GetXaxis().SetBinLabel(2, f"truth #eta < {etaCut:.2f}")
    h_summary.GetXaxis().SetBinLabel(3, "truth denom + decay acc")
    h_summary.GetXaxis().SetBinLabel(4, f"#eta < {etaCut:.2f} + decay acc")
    h_summary.GetXaxis().SetBinLabel(5, "analysis denom")
    h_summary.GetXaxis().SetBinLabel(6, "analysis numerator")
    h_summary.GetXaxis().SetBinLabel(7, "reco lowQ2 topology")
    h_summary.GetXaxis().SetBinLabel(8, "reco lowQ2 topology + decay acc")

    den_truth = 0.0
    num_truth = 0.0
    den_truth_decayAcc = 0.0
    num_truth_decayAcc = 0.0
    den_analysis = 0.0
    num_analysis = 0.0
    num_reco_lowq2_topology = 0.0
    num_reco_lowq2_topology_decayAcc = 0.0

    nEntries = tr.GetEntries()
    for i in range(nEntries):
        tr.GetEntry(i)
        #print(i)
        # Validity flag: truth branches are reset to -999 when no truth event was identified
        hasTruthScatteredElectron = (true_scat_eta[0] > -100.0 and true_scat_E[0] > 0.0)
        if not hasTruthScatteredElectron:
            continue

        decayLeptonsCentral = (abs(true_eplus_eta[0]) < 3.5 and abs(true_decay_eminus_eta[0]) < 3.5)
        reachesLowQ2Tagger = (true_scat_eta[0] < etaCut)
        inAnalysisDenom = (not requireDecayLeptonsCentral or decayLeptonsCentral)

        den_truth += 1.0
        h_eta_den.Fill(true_scat_eta[0])
        if reachesLowQ2Tagger:
            num_truth += 1.0
            h_eta_num.Fill(true_scat_eta[0])

        if decayLeptonsCentral:
            den_truth_decayAcc += 1.0
            if reachesLowQ2Tagger:
                num_truth_decayAcc += 1.0

        if inAnalysisDenom:
            den_analysis += 1.0
            if reachesLowQ2Tagger:
                num_analysis += 1.0

            eff_vs_Q2.Fill(reachesLowQ2Tagger, true_Q2[0])
            eff_vs_W.Fill(reachesLowQ2Tagger, true_W[0])
            eff_vs_yVM.Fill(reachesLowQ2Tagger, true_yUpsilon[0])
            eff_vs_E.Fill(reachesLowQ2Tagger, true_scat_E[0])
            eff_vs_pT.Fill(reachesLowQ2Tagger, true_scat_pT[0])

        if hasRecoTopology:
            recoLowQ2Topology = (reco_topology[0] == 2 or reco_topology[0] == 4)
            if recoLowQ2Topology:
                num_reco_lowq2_topology += 1.0
                if decayLeptonsCentral:
                    num_reco_lowq2_topology_decayAcc += 1.0

    h_summary.SetBinContent(1, den_truth)
    h_summary.SetBinContent(2, num_truth)
    h_summary.SetBinContent(3, den_truth_decayAcc)
    h_summary.SetBinContent(4, num_truth_decayAcc)
    h_summary.SetBinContent(5, den_analysis)
    h_summary.SetBinContent(6, num_analysis)
    h_summary.SetBinContent(7, num_reco_lowq2_topology)
    h_summary.SetBinContent(8, num_reco_lowq2_topology_decayAcc)

    fout.cd()
    h_summary.Write()
    h_eta_den.Write()
    h_eta_num.Write()
    eff_vs_Q2.Write()
    eff_vs_W.Write()
    eff_vs_yVM.Write()
    eff_vs_E.Write()
    eff_vs_pT.Write()
    
    # Save parameters
    ROOT.TParameter('double')("etaCut", etaCut).Write()
    ROOT.TParameter('double')("requireDecayLeptonsCentral", 1.0 if requireDecayLeptonsCentral else 0.0).Write()
    fout.Close()

    print("\nLow-Q2 tagger reach using truth scattered electron geometry")
    print(f"etaCut = {etaCut}, requireDecayLeptonsCentral = {requireDecayLeptonsCentral}")
    PrintEff("inclusive truth reach", den_truth, num_truth)
    PrintEff("truth reach, decay leptons central", den_truth_decayAcc, num_truth_decayAcc)
    PrintEff("analysis-denominator truth reach", den_analysis, num_analysis)
    
    if hasRecoTopology:
        PrintEff("current reco low-Q2 topology fraction, inclusive truth denom", den_truth, num_reco_lowq2_topology)
        PrintEff("current reco low-Q2 topology fraction, decay-acc truth denom", den_truth_decayAcc, num_reco_lowq2_topology_decayAcc)
        print("NOTE: reco_topology==2/4 is not a pure tagger reach efficiency; it includes the current Upsilon combinatorics and cuts.")
    
    print(f"Output written to: {outFile}")
    fin.Close()

# ==============================================================================
# SWAN / JUPYTER EXECUTION BLOCK
# ==============================================================================
# Define your parameters here instead of using command-line arguments.
# Make sure the input_file points to a valid file on EOS or your CERNBox.

input_file = "~/merged_elecs_test_tuples.root" # <-- UPDATE THIS PATH
output_file = "~/LowQ2ScatteredElectronReach.root"
eta_cut_val = -4.0
require_central = False

# Run the analysis
LowQ2ScatteredElectronReach_FromTuple(
    tupleFile=input_file,
    outFile=output_file,
    etaCut=eta_cut_val,
    requireDecayLeptonsCentral=require_central
)